# Demo 07 - Benford's Law

Benford's Law is an interesting numerical phenomenon relating to the first digit in a sequence of numbers. This code is based off of [a Python 2 recipe](https://code.activestate.com/recipes/577431-benfords-law-demo/).

In [ ]:
import sklearn.datasets
from bs4 import BeautifulSoup
import requests
import csv
import pyodbc
import pandas as pd
import toml

config = toml.load("../config.toml")
server = config["database"]["server"]
database = config["database"]["name"]
username = config["database"]["username"]
password = config["database"]["password"]
conn = pyodbc.connect('DRIVER={ODBC Driver 17 for SQL Server};SERVER='+server+';DATABASE='+database+';UID='+username+';PWD='+password)

In [ ]:
import re
import matplotlib.pyplot as plt

from math import log10

def plot_benford(iterable):
    """Plot leading digit distribution in a string iterable.
    """

    numbers = [float(n) for n in range(1, 10)]

    # Plot the frequencies as predicted by the law.
    benford = [log10(1 + 1 / d) for d in numbers]
    plt.plot(numbers, benford, 'ro', label = "Predicted")

    # Plot the actual digit frequencies.
    data = list(digits(iterable))
    plt.hist(data, range(1, 11), align = 'left', density = True,
              rwidth = 0.7, label = "Actual")

    # Set plot parameters and show it in a window.
    plt.title("Benford's Law")
    plt.xlabel("Digit")
    plt.ylabel("Frequency")

    plt.xlim(0, 10)
    plt.xticks(numbers)
    plt.legend()

    plt.show()

def digits(iterable):
    """Yield leading digits of number-like strings in an iterable.
    """

    numexp = re.compile(r'\d+(\.\d+)?([eE]\d+)?')
    leading = set("123456789")

    for item in iterable:
        for match in numexp.finditer(str(item)):
            for digit in match.group(0):
                if digit in leading:
                    yield int(digit)
                    break

## Scikit-Learn Datasets

Let's start out by looking at a couple of data sets given to us in the <code>sklearn</code> library.

### California Housing

This is a set of house prices collected in California.

In [ ]:
california = sklearn.datasets.fetch_california_housing()
plot_benford(california['data'])

This data set fits Benford's Law pretty well, alhtough we see spikes at 1 and 3 higher than the expectation.

### Wine Recognition Dataset

This is a set of characteristics around different types of wine, with values ranging from 0.48 to 1680.

In [ ]:
wine = sklearn.datasets.load_wine()
plot_benford(wine['data'])

This data set doesn't follow Benford's Law all that well, though you can still see some observation in the breach.

### The Iris dataset

This is the famous Iris dataset. The data column includes four measures: sepal length, sepal width, petal length, and petal width. These range from 0.1 to 7.9.

In [ ]:
iris = sklearn.datasets.load_iris()
plot_benford(iris['data'])

This doesn't fit a Benford distribution at all.

## Reading Data from Online Data Sets

Next, we will try reading some data from a variety of online data sets.

### List of Urban Parks by Size

Let's start by reading some data from Wikipedia using the Beautiful Soup package.

In [ ]:
data = []

headers = {'User-Agent': 'Mozilla/5.0'}
website_url = requests.get(url='https://en.wikipedia.org/wiki/List_of_urban_parks_by_size', headers=headers).text
soup = BeautifulSoup(website_url, 'html.parser')
table = soup.find_all('table', {'class':'wikitable'})[0]
for tr in table.find_all("tr"):
    for tag in tr(['span', 'sup']):
        tag.decompose()
    td = tr.find_all("td")
    if len(td) == 6:
        c = td[3].find(text=True).replace(',','').replace('\n','')
        data.append(c)
        if (c.isnumeric()):
            data.append(c)

plot_benford(data)

We can see the rough pattern, but there's an excessive number of records starting with a 1 and a deficiency in 3-9.

### List of Highest Towns by Country (m)

Here is a list of the height (in meters) of the highest town in each country.

In [ ]:
data = []

headers = {'User-Agent': 'Mozilla/5.0'}
website_url = requests.get(url='https://en.wikipedia.org/wiki/List_of_highest_towns_by_country', headers=headers).text
soup = BeautifulSoup(website_url, 'html.parser')
table = soup.find_all('table', {'class':'wikitable'})[0]
for tr in table.find_all("tr"):
    for tag in tr(['span', 'sup']):
        tag.decompose()
    td = tr.find_all("td")
    if len(td) == 7:
        c = td[4].find(text=True).replace(',','').replace('\n','')
        data.append(c)
        if (c.isnumeric()):
            data.append(c)

plot_benford(data)

We can see again that 1 is over-represented, but it still looks a bit like a Benford distribution.

### North Carolina Populations

The US Census Bureau keeps track of population estimates in addition to the decennial census. Let's take a look at North Carolina's data.

In [ ]:
ncpop = pd.read_csv('https://www2.census.gov/programs-surveys/popest/datasets/2010-2019/cities/totals/sub-est2019_37.csv')

Let's look at the 2010 census data.

In [ ]:
plot_benford(ncpop["CENSUS2010POP"])

And the 2019 estimate:

In [ ]:
plot_benford(ncpop["POPESTIMATE2019"])

Now here it is across the entire United States.

In [ ]:
uspop = pd.read_csv("https://www2.census.gov/programs-surveys/popest/datasets/2010-2019/cities/totals/sub-est2019_all.csv", encoding="ISO-8859-1")
plot_benford(uspop["CENSUS2010POP"])

### Wake County Expenditures

Finally, let's take a look at a real expenses data set. This is a data set for Wake County, North Carolina. The Wake Accountability Tax Check (WATCH) portal is an open data portal which shows financial transactions for the current and prior fiscal years. I've saved a copy of this data as captured on 2020-11-29 and converted it to a pipe-delimited text file. Note that if you run this cell on your own, the file is approximately 95 MB in size.

In [ ]:
wake_expenses = pd.read_csv("https://cspolybasepublic.blob.core.windows.net/cstrainingpublicdata/WATCH_Prior.csv", delimiter="|")

In [ ]:
plot_benford(wake_expenses["Budgeted Amount"])

We can see that the budgeted amount sort of follows a Benford distribution, though the numbers skew a bit high.

In [ ]:
plot_benford(wake_expenses["Actual Amount"])

Actual numbers align even closer to a Benford distribution. We still see an under-use of 1s an over-use of 8s &amp; 9s, but it's pretty close.

## Bus Expenses Data

Now here's what it looks like for our transit:

In [ ]:
line_items = pd.read_sql("""SELECT
    li.LineItemID,
    c.FirstDayOfMonth,
    c.CalendarYear,
    c.CalendarMonth,
    b.BusID,
    b.AverageRoadConditions,
    b.BusModelID,
    bm.BusModel,
    bm.ModelQuality,
    DATEDIFF(YEAR, b.DateFirstInService, li.LineItemDate) AS NumberOfYearsInService,
    ec.ExpenseCategoryID,
    ec.ExpenseCategory,
    v.VendorID,
    v.VendorName,
    e.EmployeeID,
    CONCAT(e.FirstName, ' ', e.LastName) AS EmployeeName,
    CONCAT(ce.FirstName, ' ', ce.LastName) AS CountersignerEmployeeName,
    li.Amount
FROM dbo.LineItem li
    INNER JOIN dbo.Calendar c
        ON li.LineItemDate = c.Date
    INNER JOIN dbo.Bus b
        ON li.BusID = b.BusID
    INNER JOIN dbo.BusModel bm
        ON b.BusModelID = bm.BusModelID
    INNER JOIN dbo.ExpenseCategory ec
        ON li.ExpenseCategoryID = ec.ExpenseCategoryID
    INNER JOIN dbo.Vendor v
        ON li.VendorID = v.VendorID
    INNER JOIN dbo.Employee e
        ON li.EmployeeID = e.EmployeeID
    LEFT OUTER JOIN dbo.Employee ce
        ON li.CountersignerEmployeeID = ce.EmployeeID;""", conn)
line_items['FirstDayOfMonth'] = pd.to_datetime(line_items['FirstDayOfMonth'])

plot_benford(line_items["Amount"])

As a first approach, this fits pretty well.  Now let's look at what we have for Glass and Sons.

In [ ]:
plot_benford(line_items.query("VendorName.str.startswith('Glass and Sons')")['Amount'])

Note that we have a bit of wonkiness here. Let's compare to three vendors at random: Comfort Rider, Safety First, and Bus Repair Shack.

In [ ]:
plot_benford(line_items.query("VendorName.str.startswith('Comfort Rider')")['Amount'])

In [ ]:
plot_benford(line_items.query("VendorName.str.startswith('Safety First')")['Amount'])

In [ ]:
plot_benford(line_items.query("VendorName.str.startswith('Bus Repair Shack')")['Amount'])

Despite the name, "Benford's Law" isn't a guarantee of anything. But we do see it observed more often than not.